<a href="https://colab.research.google.com/github/gonzaloangaut/NeuralNetworks/blob/main/Practicos/tutorial_1_pytorch_tensores_Angaut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tutorial 1: PyTorch - Tensores

Instalamos PyTorch

In [ ]:
!pip3 install torch torchvision torchaudio torchviz

  Preparing metadata (setup.py) ... done
  Created wheel for torchviz: filename=torchviz-0.0.2-py3-none-any.whl size=4133 sha256=fc61ed2747f621ea02f2f73b56ebbaf2dd6b433dcd5beea4f364e144b47d9b44
  Stored in directory: /root/.cache/pip/wheels/4c/97/88/a02973217949e0db0c9f4346d154085f4725f99c4f15a87094
Successfully built torchviz


Importamos librerias de Python

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import scipy.linalg as linalg
import sklearn as skl

Importamos librerías de PyTorch

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose
from torchviz import make_dot
import torch.optim as optim

Podemos crear tensores a partir de listas anidadas

In [ ]:
data = [[1, 2],[3, 4]] # RAM del CPU
x_data = torch.tensor(data) # Copiamos los datos de la RAM del CPU a la "RAM" de la GPU
x_data

tensor([[1, 2],
        [3, 4]])

... o a partir de numpy arrays

In [ ]:
np_array = np.array(data) # CPU
x_np = torch.from_numpy(np_array) # GPU
x_np

tensor([[1, 2],
        [3, 4]])

...o crearlos con valores predefinidos, respetando la forma y tipo de otros tensores prexistentes

In [ ]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
x_ones

tensor([[1, 1],
        [1, 1]])

In [ ]:
x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
x_rand

tensor([[0.0751, 0.1908],
        [0.2788, 0.7745]])

Recordar que shape determina el número y tamaño de las dimensiones del tensor

In [ ]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.7829, 0.0114, 0.0145],
        [0.2701, 0.2518, 0.7852]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


Como con los arrays de numpy, además de shape y dtype (data type) los tensores de PyTorch tienen el atributo device, el cual especifica en que dispositivo (i.e. en que CPU o GPU) está instanciado el tensor.
Es decir, este atributo es relevante sólo cuando trabajamos con GPUs o en un cluster de computadoras.

In [ ]:
tensor = torch.rand(3,4)
print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


Operaciones con tensores.
Podemos mover o copiar un tensor desde la CPU a la GPU.

In [ ]:
if torch.cuda.is_available():
    tensor = tensor.to('cuda')

En PyTorch el manejo de las componentes de los tensores (escalares, vectoriales, o sub-tensoriales) es equivalente al de numpy.

In [ ]:
tensor = torch.ones(4, 4)
print('First row: ', tensor[0])
print('First column: ', tensor[:, 0])
print('Last column:', tensor[:, -1])
print('Last column:', tensor[..., -1])
tensor[:,1] = 0
print(tensor)

First row:  tensor([1., 1., 1., 1.])
First column:  tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


Podemos concatenar tensores (ver también `torch.stack` que trabaja de manera similar)

In [ ]:
t1 = torch.cat([tensor, 2*tensor, 3*tensor], dim=0)
print(t1)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [2., 0., 2., 2.],
        [2., 0., 2., 2.],
        [2., 0., 2., 2.],
        [2., 0., 2., 2.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.],
        [3., 0., 3., 3.]])


In [ ]:
t1 = torch.cat([tensor, 2*tensor, 3*tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 2., 0., 2., 2., 3., 0., 3., 3.],
        [1., 0., 1., 1., 2., 0., 2., 2., 3., 0., 3., 3.],
        [1., 0., 1., 1., 2., 0., 2., 2., 3., 0., 3., 3.],
        [1., 0., 1., 1., 2., 0., 2., 2., 3., 0., 3., 3.]])


Las siguientes, son diferentes formas de multiplicar matricialmente dos tensores.
Aquí `y1`, `y2`, `y3` resultarán con el mismo valor.

In [ ]:
y1 = tensor @ tensor.T

y2 = tensor.matmul(tensor.T)

y3 = torch.rand_like(tensor)
torch.matmul(tensor, tensor.T, out=y3)

tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])

De manera similar, estas son diferentes formas de multiplicar punto a punto dos tensores.
Aquí `z1`, `z2`, `z3` resultarán con el mismo valor.

In [ ]:
z1 = tensor * tensor

z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

Podemos, por ejemplo, sumar todas las componentes de un tensor para generar un tensor de una única componente.

In [ ]:
agg = tensor.sum()
print(agg,type(agg)) # Esto es un tensor de PyTorch de una sola componente.

tensor(12.) <class 'torch.Tensor'>


Para acceder al valor numérico en formato Python de dicha componente, usamos el método `.item()`.

In [ ]:
agg_item = agg.item()
print(agg_item,type(agg_item)) # Esto es (algo así como) un número flotante de Python.

12.0 <class 'float'>


In place operations: las funciones miembreo de un tensor que terminan en el caracter `_` corresponden a operaciones que actuan *in place*, i.e. que modifican
las componentes del tensor sin generar una copia.
Por ejemplo, la siguiente expresión suma 5 a cada componente del tensor llamado `tensor`.

In [ ]:
tensor.add_(5)
tensor

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])

Se puede crear una *vista* (view) en formato numpy de un tensor de PyTorch.

In [ ]:
t = torch.ones(5)
n = t.numpy()
print(f"t: {t}")
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


Notar que `n` no es una copia del contenido de `t`, sinó un view.
Entonces, un cambio en el tensor `t` se ve reflejado en `n`.

In [ ]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]
